In [ ]:
import subprocess
import json
import os
from hipporag import HippoRAG


 # Decrypt and load environment variables from dotenvx
result = subprocess.run(['dotenvx', 'get', '--format', 'json'], capture_output=True,
text=True)

if result.returncode == 0:
    env_vars = json.loads(result.stdout)
    for key, value in env_vars.items():
        os.environ[key] = value
    print("✅ dotenvx variables loaded successfully!")
else:
    print(f"❌ Error loading dotenvx variables: {result.stderr}")

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from src.config import Config
from src.clients import get_clients
from src.utils.documents import process_pptx_file, fuse_strings
from src.pipeline import run_pipeline, compare_runs

# Setup: clients and corpus

In [ ]:
Config.setup_env()
Config.setup_directories()

clients = get_clients()

directory_path = "./documents_to_index"
file_paths = [
    os.path.join(directory_path, f)
    for f in os.listdir(directory_path)
    if os.path.isfile(os.path.join(directory_path, f))
]

contents = process_pptx_file(file_paths)

# Build chunking strategies

In [ ]:
content_lists = {
    "baseline":  contents,
    "fuse_1000": fuse_strings(contents, 256),
    "fuse_2000": fuse_strings(contents, 512),
    "fuse_4000": fuse_strings(contents, 1024),
}

for label, cl in content_lists.items():
    print(f"{label}: {len(cl)} chunks")

# Run full pipeline per strategy


In [ ]:
all_runs = {}
for label, cl in content_lists.items():
    print(f"\n=== Running pipeline: {label} ({len(cl)} chunks) ===")
    all_runs[label] = await run_pipeline(
        cl, label, clients, "./templates/benchmark.csv"
    )

# Compare strategies across chunking granularity

In [ ]:
comparison_df = compare_runs(all_runs)
comparison_df